# Montage Management

In [ ]:
import re
import os
# Import pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.plot_data.plotter import *
from pyologger.utils.montage_manager import MontageManager
from pyologger.utils.param_manager import ParamManager
from pyologger.load_data.datareader import DataReader
from pyologger.load_data.metadata import Metadata

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()

### Fetch metadata

Load in metadata stored in Notion databases. Alternatively, load in your own metadata in separate dataframes for deployments, loggers, recordings, animals, and datasets. See examples here in the `metadata_snapshot.pkl` file.

In [ ]:
from datetime import datetime, timedelta

overwrite = True
# Define the path to the metadata pickle file
metadata_pickle_path = os.path.join(data_dir, "00_Metadata/metadata_snapshot.pkl")

# Check if the metadata pickle file exists and is more than 5 days old
if overwrite or not os.path.exists(metadata_pickle_path) or (datetime.now() - datetime.fromtimestamp(os.path.getmtime(metadata_pickle_path))) > timedelta(days=5):
    
    metadata = Metadata()

    # Save database variables
    deployment_db = metadata.get_metadata("deployment_DB")
    logger_db = metadata.get_metadata("logger_DB")
    recording_db = metadata.get_metadata("recording_DB")
    animal_db = metadata.get_metadata("animal_DB")
    dataset_db = metadata.get_metadata("dataset_DB")
    procedure_db = metadata.get_metadata("procedure_DB")
    observation_db = metadata.get_metadata("observation_DB")
    collaborator_db = metadata.get_metadata("collaborator_DB")
    location_db = metadata.get_metadata("location_DB")
    montage_db = metadata.get_metadata("montage_DB")
    signal_db = metadata.get_metadata("signal_DB")
    attachment_db = metadata.get_metadata("attachment_DB")
    originalchannel_db = metadata.get_metadata("originalchannel_DB")
    standardizedchannel_db = metadata.get_metadata("standardizedchannel_DB")
    derivedsignal_db = metadata.get_metadata("derivedsignal_DB")
    derivedchannel_db = metadata.get_metadata("derivedchannel_DB")

    # Get the relations map
    relations_map = metadata.relations_map

    # Define the path to save the relations map
    relations_map_path = os.path.join(config['paths']['local_repo_path'], 'relations_map.json')

    # Save the relations map as a JSON file
    with open(relations_map_path, 'w') as file:
        json.dump(relations_map, file, indent=4)

    print(f"Relations map saved at: {relations_map_path}")

    ## OPTIONAL: Save metadata snapshot as a pickle file
    metadata.notion = None  # Temporarily remove the Notion client
    # Save metadata snapshot as a pickle file
    with open(metadata_pickle_path, "wb") as file:
        pickle.dump(metadata, file)

    print(f"Metadata snapshot saved at: {metadata_pickle_path}")
else:
    print(f"Recent metadata snapshot loaded, already present at: {metadata_pickle_path}")
    
    # Load the metadata snapshot from the pickle file
    with open(metadata_pickle_path, "rb") as file:
        metadata = pickle.load(file)
    
    # Save database variables
    deployment_db = metadata.get_metadata("deployment_DB")
    logger_db = metadata.get_metadata("logger_DB")
    recording_db = metadata.get_metadata("recording_DB")
    animal_db = metadata.get_metadata("animal_DB")
    dataset_db = metadata.get_metadata("dataset_DB")
    procedure_db = metadata.get_metadata("procedure_DB")
    observation_db = metadata.get_metadata("observation_DB")
    collaborator_db = metadata.get_metadata("collaborator_DB")
    location_db = metadata.get_metadata("location_DB")
    montage_db = metadata.get_metadata("montage_DB")
    signal_db = metadata.get_metadata("signal_DB")
    attachment_db = metadata.get_metadata("attachment_DB")
    originalchannel_db = metadata.get_metadata("originalchannel_DB")
    standardizedchannel_db = metadata.get_metadata("standardizedchannel_DB")
    derivedsignal_db = metadata.get_metadata("derivedsignal_DB")
    derivedchannel_db = metadata.get_metadata("derivedchannel_DB")


In [ ]:
# Select dataset folder
dataset_folder = select_folder(data_dir, "Select a dataset folder:")

In [ ]:
deployment_folder = select_folder(dataset_folder, "Select a deployment folder:")

In [ ]:
# Extract deployment_id and animal_id from the folder name
# Pattern: YYYY-MM-DD_animalid-NNNN or YYYY-MM-DD_animalid-NNNN_suffix
match = re.match(r"(\d{4}-\d{2}-\d{2}_[a-z]+-\d+)", os.path.basename(deployment_folder), re.IGNORECASE)
if match:
    deployment_id = match.group(1)  # Extract YYYY-MM-DD_animalID
    animal_id = deployment_id.split("_")[1]  # Extract animal ID (everything after first _)
    print(f"✅ Extracted deployment ID: {deployment_id}, Animal ID: {animal_id}")
else:
    raise ValueError(f"❌ Unable to extract deployment ID from folder: {deployment_folder}")

## Read Files in Deployment Folder

Using datareader to load in files


In [ ]:
# Print extracted values for debugging
print(f"🐳 Deployment ID: {deployment_id}, Animal ID: {animal_id}")

deployment_info, loggers_used = metadata.extract_essential_metadata(deployment_id)

In [ ]:
# Step 4: Initialize DataReader with dataset folder, deployment ID, and optional data subfolder
data_pkl = DataReader(dataset_folder=dataset_folder, deployment_id=deployment_id, data_subfolder="01_raw-data", montage_path=montage_path)

# Step 5: Initialize config manager
param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)

## Preview montages

In [ ]:
loggers_used

In [ ]:
logger_no = 0  # Change this index to select different loggers
loggers_used[logger_no]['Montage ID']

In [ ]:
original_channels = originalchannel_db[originalchannel_db['Montages'].str.contains(loggers_used[logger_no]['Montage ID'], na=False)]
original_channels

In [ ]:
print(standardizedchannel_db)

In [ ]:
standardizedchannel_db

In [ ]:
signal_db

In [ ]:
use_csv = False # 🔁 Set this to True if you want to load montage from CSVs
csv_folder = "/path/to/csvs"  # Folder where your CSVs are stored

montage_inputs = {}

standardizedchannel_db = standardizedchannel_db.copy()
# Create Color Default column: use Color Override if present, else use Color
standardizedchannel_db["Color Preview"] = standardizedchannel_db.apply(
    lambda row: row["Color Override"] if pd.notna(row["Color Override"]) and row["Color Override"] != "" 
    else signal_db.loc[signal_db["page_id"] == row["Parent signal"], "Color"].values[0] 
    if not signal_db.loc[signal_db["page_id"] == row["Parent signal"], "Color"].empty 
    else None,
    axis=1
)

# Extract color code from Color Preview (e.g., from '\color {#6ca1c3} ███████' get '6ca1c3')
standardizedchannel_db.loc[:, "Color"] = standardizedchannel_db["Color Preview"].apply(
    lambda x: x[9:15] if isinstance(x, str) and len(x) >= 15 else None
)

# Create Standardized Unit column: use Unit Override if present, else use Standardized Unit from signal_db
standardizedchannel_db["Standardized Unit"] = standardizedchannel_db.apply(
    lambda row: row["Unit Override"] if pd.notna(row["Unit Override"]) and row["Unit Override"] != ""
    else signal_db.loc[signal_db["page_id"] == row["Parent signal"], "Standardized Unit"].values[0]
    if not signal_db.loc[signal_db["page_id"] == row["Parent signal"], "Standardized Unit"].empty
    else None,
    axis=1
)

standardizedchannel_db

In [ ]:
def _norm(s):
    return str(s).strip().lower()

def _build_std_expanded(std_df: pd.DataFrame) -> pd.DataFrame:
    std = std_df.copy()
    std.columns = [c.strip().lower().replace(" ", "_") for c in std.columns]

    if "channel_id" not in std.columns and "standardized_channel_id" not in std.columns:
        raise ValueError("standardizedchannel_db missing Channel ID / Standardized Channel ID")
    if "channel_id" in std.columns and "standardized_channel_id" not in std.columns:
        std["standardized_channel_id"] = std["channel_id"]

    if "parent_signal" not in std.columns:
        raise ValueError("standardizedchannel_db missing Parent signal")
    if "original_channels" not in std.columns:
        raise ValueError("standardizedchannel_db missing Original Channels")

    std["standardized_channel_id"] = std["standardized_channel_id"].astype(str).map(_norm)
    std["parent_signal"] = std["parent_signal"].astype(str).map(_norm)
    std["standardized_unit"] = std.get("standardized_unit", pd.Series([None]*len(std)))
    std["original_channels"] = std["original_channels"].fillna("").astype(str)

    std = std.assign(
        original_channel_id=std["original_channels"].str.split(",")
    ).explode("original_channel_id")
    std["original_channel_id"] = std["original_channel_id"].astype(str).map(_norm)
    std = std[std["original_channel_id"] != ""]

    return std[["original_channel_id", "standardized_channel_id", "standardized_unit", "parent_signal"]].drop_duplicates()


def _resolve_candidates(cands: pd.DataFrame, manufacturer_signal_name: str) -> pd.Series:
    if len(cands) == 1:
        return cands.iloc[0]

    msn = _norm(manufacturer_signal_name)

    # Prefer exact parent_signal == manufacturer_signal_name if available
    exact = cands[cands["parent_signal"] == msn]
    if len(exact) == 1:
        return exact.iloc[0]

    # Otherwise prefer non-corrected parent first, then deterministic sort
    ranked = cands.assign(
        is_corrected=cands["parent_signal"].str.startswith("corrected_")
    ).sort_values(["is_corrected", "parent_signal", "standardized_channel_id"])
    return ranked.iloc[0]


std_expanded = _build_std_expanded(standardizedchannel_db)
montage_inputs = {}

for logger in loggers_used:
    logger_id = logger["Logger ID"]
    montage_id = logger["Montage ID"]

    orig = originalchannel_db[
        originalchannel_db["Montages"].str.contains(montage_id, na=False)
    ].copy()
    if orig.empty:
        print(f"⚠️ No original channels found for montage ID: {montage_id} (Logger: {logger_id})")
        continue

    orig = orig.rename(columns={
        "Original Channel ID": "original_channel_id",
        "Original Unit": "original_unit",
        "Manufacturer Signal Name": "manufacturer_signal_name",
    })
    orig["original_channel_id"] = orig["original_channel_id"].astype(str).map(_norm)

    rows = []
    unresolved = []

    for _, r in orig.iterrows():
        oid = r["original_channel_id"]
        cands = std_expanded[std_expanded["original_channel_id"] == oid]
        if cands.empty:
            unresolved.append(oid)
            rows.append({
                "original_channel_id": oid,
                "original_unit": r["original_unit"],
                "manufacturer_signal_name": r["manufacturer_signal_name"],
                "standardized_channel_id": None,
                "standardized_unit": None,
                "parent_signal": None,
            })
            continue

        pick = _resolve_candidates(cands, r["manufacturer_signal_name"])
        rows.append({
            "original_channel_id": oid,
            "original_unit": r["original_unit"],
            "manufacturer_signal_name": r["manufacturer_signal_name"],
            "standardized_channel_id": pick["standardized_channel_id"],
            "standardized_unit": pick["standardized_unit"],
            "parent_signal": pick["parent_signal"],
        })

    montage_df = pd.DataFrame(rows)
    montage_inputs[logger_id] = montage_df

    if unresolved:
        print(f"⚠️ {len(unresolved)} unresolved original channels for {logger_id}: {unresolved[:10]}")
    print(f"✅ Created montage_df for Logger: {logger_id} (Montage ID: {montage_id})")


In [ ]:
montage_inputs['CD-66']

In [ ]:
montage_manager = MontageManager(montage_folder=os.path.dirname(montage_path))

montages_metadata = montage_manager.add_missing_montages_per_logger(
	loggers_used=loggers_used,
	montage_inputs=montage_inputs
)

In [ ]:
montage_inputs['PD-D3']

In [ ]:
montage_inputs['PD-03']

In [ ]:
montage_inputs['PD-D3']